# Pydantic: Data Validation & Settings Management

Pydantic is the most widely used data validation library for Python. It enforces type hints at runtime and provides user-friendly errors when data is invalid.

## 1. Why Pydantic?

| Feature | Standard Type Hints (mypy) | Pydantic |
| :--- | :--- | :--- |
| **Runtime Check** | ❌ (Static only) | ✅ (Always checked) |
| **Data Coercion** | ❌ (Manual only) | ✅ (e.g., "123" → 123) |
| **Error Reports** | ❌ (IDE only) | ✅ (Detailed JSON errors) |
| **FastAPI Integration**| ❌ | ✅ (Native) |

### Core Principle
Type hints protect you from yourself while writing code. **Pydantic protects your application from the outside world** (APIs, User Input, Databases).

In [ ]:
%pip install pydantic email-validator

## 2. Basic Models & Field Validation

Use `BaseModel` to define your data structure and `Field` to add constraints.

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    # id is required, must be > 0
    id: int = Field(gt=0)
    
    # username must be 3-20 chars
    username: str = Field(min_length=3, max_length=20)
    
    # age is optional with a default
    age: int | None = Field(default=None, ge=18, le=120)
    
    # bio has a default value
    bio: str = "No bio provided"

# Valid Data
user = User(id=1, username="alice", age="25") # Note: "25" is coerced to 25
print(f"User created: {user.model_dump()}")

try:
    # Invalid Data: id is 0, username too short
    User(id=0, username="al")
except Exception as e:
    print(f"\nValidation Error:\n{e}")

## 3. Modern Syntax: `Annotated`

`Annotated` allows you to define reusable types. This is the **preferred modern way** because it keeps your models clean.

In [ ]:
from typing import Annotated

# Define reusable types
UserID = Annotated[int, Field(gt=0)]
Username = Annotated[str, Field(min_length=3, max_length=20)]

class Product(BaseModel):
    creator_id: UserID
    name: str

class Comment(BaseModel):
    author_id: UserID
    text: str

## 4. Advanced Types & Defaults

- **`Literal`**: Restrict to specific values.
- **`EmailStr`**: Validates emails (requires `email-validator`).
- **`SecretStr`**: Masks sensitive data in logs.
- **`default_factory`**: Used for dynamic defaults (like current time or empty lists).

In [ ]:
from typing import Literal
from datetime import datetime, UTC
from pydantic import EmailStr, SecretStr, HttpUrl

class Account(BaseModel):
    email: EmailStr
    password: SecretStr
    status: Literal["active", "pending", "banned"] = "pending"
    website: HttpUrl | None = None
    
    # Use default_factory for mutable types or dynamic values
    created_at: datetime = Field(default_factory=lambda: datetime.now(UTC))
    tags: list[str] = Field(default_factory=list)

acc = Account(email="test@example.com", password="shhh", status="active")
print(f"Password is masked: {acc.password}")
print(f"Actual value: {acc.password.get_secret_value()}")

## 5. Custom Validators

Sometimes standard `Field` constraints aren't enough. Use `@field_validator` for single fields and `@model_validator` for cross-field logic.

In [ ]:
from pydantic import field_validator, model_validator

class Signup(BaseModel):
    username: str
    password: str
    confirm_password: str

    @field_validator("username")
    @classmethod
    def no_spaces(cls, v: str) -> str:
        if " " in v:
            raise ValueError("Username cannot contain spaces")
        return v.lower()

    @model_validator(mode="after")
    def check_passwords_match(self) -> "Signup":
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

try:
    Signup(username="cool user", password="123", confirm_password="123")
except ValueError as e:
    print(f"Error: {e}")

## 6. Model Configuration (`ConfigDict`)

Control global model behavior like immutability (`frozen`), strict mode, and field aliasing.

In [ ]:
from pydantic import ConfigDict

class ImmutableUser(BaseModel):
    model_config = ConfigDict(
        frozen=True,               # Cannot change values after creation
        populate_by_name=True,     # Allow using aliases
        extra="forbid"             # Disallow unknown fields
    )
    
    username: str = Field(alias="user_name")

user = ImmutableUser(user_name="bob")
print(f"Accessed via attribute: {user.username}")

try:
    user.username = "alice"
except Exception as e:
    print(f"Error modifying frozen model: {type(e).__name__}")

## 7. Serialization & Computed Fields

Convert models back to dictionaries or JSON, and add "virtual" fields that are calculated on the fly.

In [ ]:
from pydantic import computed_field

class Circle(BaseModel):
    radius: float

    @computed_field
    @property
    def area(self) -> float:
        import math
        return math.pi * (self.radius ** 2)

c = Circle(radius=5)
print(f"JSON Output: {c.model_dump_json(indent=2)}")
print(f"Dict Output (excluding area): {c.model_dump(exclude={'area'})}")

### Summary Cheat Sheet
- **`BaseModel`**: The foundation for all Pydantic models.
- **`Field`**: Define constraints (min/max, regex, defaults).
- **`Annotated`**: Create reusable, validated types.
- **`model_dump()`**: Convert to `dict`.
- **`model_validate_json()`**: Load from `JSON` string.
- **`ConfigDict`**: Set model-wide rules (strict, frozen, extra).